In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import sys
sys.path.append('..')
from modules.source import OUTPUT_FOLDER_PATH, parquet_daily, Sources
from modules.helpers import add_regime_shading

df_combined_multiindex = pd.read_parquet(parquet_daily)
prices = df_combined_multiindex.xs("price", axis=1, level=1)
prices.index.name = "Year"

plt.rcParams.update({ # set default settings
   "figure.figsize": (10, 4),  
   "figure.dpi": 300,
   "font.size": 8,
   "font.family": "sans-serif",

   "axes.grid": True,
   "grid.linestyle": "--",
   "grid.alpha": 0.5,
   "lines.linewidth": 1,
   "lines.color": "#1f77b4",
   "axes.titlesize": 12,
   "axes.titleweight": "bold",
   "axes.titley": 1.05,
   "savefig.bbox": "tight",  # Prevent cropped labels on export
})

## 1. Equities

The overlay ("Equities – KLCI & Bursa Sectors") uses a log y-axis, so that equal percentage moves take up equal vertical distance regardless of level, a series that's barely grown (KLCI) and one that's grown roughly 30x (Technology) are both legible on the same chart.

In [ ]:
bursa_cols = [Sources.KLCI, Sources.financials, Sources.plantation, Sources.REITs, Sources.technology, Sources.industrial_products]

bursa_df = prices[bursa_cols].apply(lambda col: (col / col.dropna().iloc[0]) * 100)

ax = bursa_df[bursa_cols].plot(ylabel="Index Level (100 = Start)", logy=True, rot=0)
plt.setp(ax.get_xticklabels(), ha="center") # center the year labels

ax.set_title("Equities - KLCI & Bursa Sectors")
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:g}"))
ax.spines[["top", "right"]].set_visible(False)

# show min and max value on y-scale 
global_min = round(bursa_df.min().min(), 1)
global_max = round(bursa_df.max().max(), 1)
ax.set_yticks([global_min, 100, 500, 1000, global_max])
ax.axhline(global_min, color="black", linestyle=":", alpha=0.8)
ax.axhline(global_max, color="black", linestyle=":", alpha=0.8)

# shade interest rate regime periods 
add_regime_shading(ax, show_covid=True)

plt.savefig(f"{OUTPUT_FOLDER_PATH}/descriptive_charts/equities.png", bbox_inches="tight")
plt.show()

Even on a log scale, all other sectors' price patterns are muted by technology sector's parabolic growth. We give each series its own independently-scaled panel for the full detail of each series' own trajectory patterns that the crowded overlay can't show. The two figures answer different questions.

Note: Plantation's line only starts around late 2017 rather than at the 2015 sample start. That's the SD Guthrie Berhad listing-date constraint.

In [ ]:
axes = bursa_df.plot(subplots=True, layout=(2, 3), sharex=True, title="KLCI & Bursa Sectors Performance")

for ax in axes.flat:
   if legend := ax.get_legend():
      legend.set_frame_on(False)

plt.savefig(f"{OUTPUT_FOLDER_PATH}/descriptive_charts/equities_grid.png", bbox_inches="tight")
plt.show()

## 2. Commodities
Brent Oil and Crude Palm Oil (PPOILUSDM) are rebased to 100 rather than plotted in native units (USD/barrel vs. USD/tonne). Brent renders as a continuous daily line and CPO renders as a step function, because PPOILUSDM is IMF's monthly series held flat within each month.

In [ ]:
comm_cols = [Sources.brent_oil, Sources.palm_oil_global]

comm_df = prices[comm_cols].apply(lambda col: (col / col.dropna().iloc[0]) * 100)

ax = comm_df.plot(ylabel="Rebased (100 = Start)", rot=0)
plt.setp(ax.get_xticklabels(), ha="center")

ax.set_title("Commodities")
ax.legend( ["Brent Oil", "Crude Palm Oil (PPOILUSDM)"], frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)

# Reference baseline & Macro Shading
ax.axhline(100, color="#888888", linestyle="--", linewidth=0.8, alpha=0.7)
add_regime_shading(ax, show_covid=True)

plt.savefig(f"{OUTPUT_FOLDER_PATH}/descriptive_charts/commodities.png", bbox_inches="tight")
plt.show()

## 3. Policy Rates

In [ ]:
df_rates = prices[[Sources.FFR_midpoint, Sources.OPR, Sources.UST_10Y]].dropna()

ax = df_rates.plot(title="Central Bank Policy Rates & US Treasury Yield", ylabel="Rate (%)", color=["#1f77b4", "#d62728", "orange"], rot=0)
plt.setp(ax.get_xticklabels(), ha="center")

ax.legend(["US Fed Funds Rate (FFR)", "BNM Overnight Policy Rate (OPR)", "10-Y US Treasury Yield"], frameon=True, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
add_regime_shading(ax, show_ir=True)

plt.savefig(f"{OUTPUT_FOLDER_PATH}/descriptive_charts/policy_rates_UST.png", bbox_inches="tight")
plt.show()

## 4. FX & Risk Backdrop
The dashed "Stress Level (30)" line on the VIX panel is a standard market reference point (VIX above ~30 is conventionally read as elevated stress).

In [ ]:
rebase_cols = [Sources.USDMYR, Sources.DXY]
df_fx = prices[rebase_cols + [Sources.VIX]].dropna()

df_rebased = (df_fx[rebase_cols] / df_fx[rebase_cols].iloc[0]) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(df_rebased.index, df_rebased[Sources.USDMYR], label="USD/MYR", color="#2ca02c")
ax1.plot( df_rebased.index, df_rebased[Sources.DXY], label="US Dollar Index (DXY)", color="#1f77b4")
ax1.set_ylabel("Index (100 = Start)")
ax1.set_title("FX & Risk Backdrop")
ax1.legend(frameon=False, loc="upper left")
ax1.spines[["top", "right"]].set_visible(False)
add_regime_shading(ax1, show_ir=True)

ax2.plot( df_fx.index, df_fx[Sources.VIX], label="VIX Index", color="#d62728")
ax2.axhline(30, color="black", linestyle="--", alpha=0.7, label="Stress Level (30)")
ax2.set_ylabel("VIX Level")
ax2.legend(frameon=False, loc="upper left")
ax2.spines[["top", "right"]].set_visible(False)
add_regime_shading(ax2, show_covid=True)

plt.setp(ax2.get_xticklabels(), rotation=0, ha="center")

plt.savefig(f"{OUTPUT_FOLDER_PATH}/descriptive_charts/fx_risk_backdrop.png", bbox_inches="tight")
plt.show()